# Prepare Training Data

This notebook prepares the matched and pointwise-labeled history trajectories for model training.

The workflow:

1. Load all seasonal matched datasets for one airport.
2. Combine the seasons into one airport dataset.
3. Preserve the previous flight IDs and assign new unique flight IDs.
4. Keep only ground and near-ground movement up to the last relevant airport point.
5. Compute the remaining trajectory features required for training.
6. Store the prepared airport dataset.


## 1. Imports

In [ ]:
from pathlib import Path
import warnings

import pandas as pd
from traffic.core import Traffic
from traffic.data import airports

warnings.filterwarnings("ignore", category=RuntimeWarning)


## 2. Configuration

In [ ]:
airport_str = "LSZH"

seasons = [
    "Spring",
    "Summer",
    "Autumn",
    "Winter",
]

dataset_version = "V1"
max_workers = 10

PROJECT_ROOT = Path.home() / "Projects" / "OSN_pushback_detection"
DATA_ROOT = PROJECT_ROOT / "data"

matched_root = (
    DATA_ROOT
    / "train_validation"
    / "04_matched"
)

prepared_dir = (
    DATA_ROOT
    / "train_validation"
    / "05_final"
    / airport_str
)
prepared_dir.mkdir(parents=True, exist_ok=True)

prepared_path = (
    prepared_dir
    / f"{airport_str}_{dataset_version}.parquet"
)

if airports[airport_str] is None:
    raise ValueError(f"Airport '{airport_str}' is not available in traffic.data.airports.")

print(f"Airport:       {airport_str}")
print(f"Seasons:       {', '.join(seasons)}")
print(f"Output:        {prepared_path}")


## 3. Load Seasonal Matched Datasets

Load the matched datasets generated by `03_match_history` for all configured seasons.

In [ ]:
seasonal_trajectories = []

for season in seasons:
    matched_path = (
        matched_root
        / airport_str
        / season
        / f"{airport_str}_{season}_matched.parquet"
    )

    if not matched_path.exists():
        raise FileNotFoundError(
            f"Matched dataset not found: {matched_path}"
        )

    data = pd.read_parquet(matched_path)
    seasonal_trajectories.append(Traffic(data))

    print(
        f"Loaded {season}: "
        f"{data['flight_id'].nunique():,} flights, "
        f"{len(data):,} points"
    )


## 4. Combine Seasons

In [ ]:
trajs = seasonal_trajectories[0]

for seasonal_trajs in seasonal_trajectories[1:]:
    trajs = trajs + seasonal_trajs

print(
    f"Combined dataset: "
    f"{trajs.data['flight_id'].nunique():,} current flight IDs, "
    f"{len(trajs.data):,} points"
)

trajs


## 5. Assign Unique Flight IDs

Flight IDs generated independently for each season may overlap. Preserve the previous ID and assign a new unique ID after combining all seasons.

`assign_id()` intentionally uses one worker to keep the resulting IDs deterministic.

In [ ]:
trajs.data["flight_id_old"] = trajs.data["flight_id"]

trajs = trajs.drop(columns=["flight_id"])

trajs = trajs.assign_id().eval(
    max_workers=1,
    desc="Assigning unique flight IDs",
)

print(
    f"Assigned {trajs.data['flight_id'].nunique():,} unique flight IDs"
)

trajs


## 6. Keep Ground and Near-Ground Movement

Keep each trajectory only up to its last point inside the airport area that is either below 500 ft above airport elevation or has no geometric altitude value.

The complete trajectory up to that point is retained.

In [ ]:
airport = airports[airport_str]
altitude_threshold = airport.altitude + 500

print(
    f"Airport elevation: {airport.altitude:.0f} ft | "
    f"Near-ground threshold: {altitude_threshold:.0f} ft"
)


In [ ]:
def trim_to_last_low_point(flight):
    clipped = flight.clip(airport.shape)

    if clipped is None:
        return None

    low = clipped.data[
        (clipped.data["geoaltitude"] < altitude_threshold)
        | clipped.data["geoaltitude"].isna()
    ]

    if low.empty:
        return None

    last_low_time = low["timestamp"].max()

    return flight.before(
        last_low_time,
        strict=False,
    )


trajs = (
    trajs
    .pipe(trim_to_last_low_point)
    .eval(
        max_workers=max_workers,
        desc="Trimming to last near-ground point",
    )
)

print(
    f"Remaining flights after trimming: "
    f"{trajs.data['flight_id'].nunique():,}"
)

trajs


## 7. Compute Remaining Features

Compute cumulative trajectory distance and unwrap the computed track so that heading changes remain continuous across the 0°/360° boundary.

In [ ]:
trajs = trajs.cumulative_distance().eval(
    max_workers=max_workers,
    desc="Computing cumulative distance",
)

trajs = trajs.unwrap("compute_track").eval(
    max_workers=max_workers,
    desc="Unwrapping computed track",
)

trajs


## 8. Validate Prepared Dataset

Perform basic checks before storing the prepared training data.

In [ ]:
required_columns = {
    "flight_id",
    "flight_id_old",
    "timestamp",
    "latitude",
    "longitude",
    "compute_track",
    "cumdist",
    "Pushback",
    "Airport",
    "Season",
}

missing_columns = required_columns - set(trajs.data.columns)

if missing_columns:
    raise ValueError(
        f"Prepared dataset is missing required columns: "
        f"{sorted(missing_columns)}"
    )

if trajs.data["flight_id"].isna().any():
    raise ValueError("Prepared dataset contains missing flight IDs.")

print(f"Flights:          {trajs.data['flight_id'].nunique():,}")
print(f"Points:           {len(trajs.data):,}")
print(f"Pushback points:  {int(trajs.data['Pushback'].sum()):,}")
print(f"Seasons present:  {sorted(trajs.data['Season'].dropna().unique())}")


## 9. Save Prepared Training Data

Store the airport-level prepared dataset for the subsequent training-data processing steps.

In [ ]:
trajs.to_parquet(prepared_path)

print(f"Saved prepared training data: {prepared_path}")
